In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
# --- Connection info ---
PGHOST = "136.112.214.240"
PGPORT = "5432"
PGDATABASE = "feature-service-db"
PGUSER = "user"
PGPASSWORD = "postgres"

# --- Create SQLAlchemy engine ---
engine = create_engine(f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# --- Test connection ---
with engine.begin() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected to:", result.scalar())


Connected to: PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by Debian clang version 12.0.1, 64-bit


In [3]:
# Load Price Data DataFrame

df = pd.read_csv('./dataframes/df_x_minute_price_data.csv')
df
#df.dtypes

,datetime,ticker,price_open,price_close,price_high,price_low,price_vol
0,2015-01-01 14:56:00,X:BTCUSD,314.02868,314.02868,314.02868,314.02868,0.220000
1,2015-01-01 15:01:00,X:BTCUSD,314.03924,314.03924,314.03924,314.03924,0.015000
2,2015-01-01 17:32:00,X:BTCUSD,324.99787,324.99787,324.99787,324.99787,0.150000
3,2015-01-01 17:39:00,X:BTCUSD,324.99703,324.99703,324.99703,324.99703,0.050000
4,2015-01-02 04:00:00,X:BTCUSD,313.17849,313.17849,313.17849,313.17849,0.010400
...,...,...,...,...,...,...,...
19614,2015-12-31 09:55:00,X:ETHUSD,0.96000,0.96664,0.96664,0.96000,10.351982
19615,2015-12-31 10:02:00,X:ETHUSD,0.94000,0.94500,0.94500,0.94000,1.200000
19616,2015-12-31 10:15:00,X:ETHUSD,0.95000,0.95000,0.95000,0.95000,1.000000
19617,2015-12-31 13:42:00,X:ETHUSD,0.96000,0.96000,0.96000,0.96000,1.000000


In [4]:
# Convert to datetime
df['datetime'] = pd.to_datetime(df['datetime'])

In [5]:
df.dtypes

datetime       datetime64[ns]
ticker                 object
price_open            float64
price_close           float64
price_high            float64
price_low             float64
price_vol             float64
dtype: object

In [6]:
# # Specific cutoff time
# cutoff = pd.Timestamp('2025-10-05 18:20:00')

# # Keep only rows at or before the cutoff
# df = df[df['datetime'] <= cutoff]

df

,datetime,ticker,price_open,price_close,price_high,price_low,price_vol
0,2015-01-01 14:56:00,X:BTCUSD,314.02868,314.02868,314.02868,314.02868,0.220000
1,2015-01-01 15:01:00,X:BTCUSD,314.03924,314.03924,314.03924,314.03924,0.015000
2,2015-01-01 17:32:00,X:BTCUSD,324.99787,324.99787,324.99787,324.99787,0.150000
3,2015-01-01 17:39:00,X:BTCUSD,324.99703,324.99703,324.99703,324.99703,0.050000
4,2015-01-02 04:00:00,X:BTCUSD,313.17849,313.17849,313.17849,313.17849,0.010400
...,...,...,...,...,...,...,...
19614,2015-12-31 09:55:00,X:ETHUSD,0.96000,0.96664,0.96664,0.96000,10.351982
19615,2015-12-31 10:02:00,X:ETHUSD,0.94000,0.94500,0.94500,0.94000,1.200000
19616,2015-12-31 10:15:00,X:ETHUSD,0.95000,0.95000,0.95000,0.95000,1.000000
19617,2015-12-31 13:42:00,X:ETHUSD,0.96000,0.96000,0.96000,0.96000,1.000000


In [7]:
# Bulk Insert to SQL Table
df.to_sql(
    name='x_min_price_data',
    con=engine,
    if_exists='append',   # options: 'fail', 'replace', 'append'
    index=False,          # don't include DataFrame index
    chunksize=1000,       # send in batches of 1000 for performance
    method='multi'        # enables bulk insert
)


19619